In [1]:
import numpy as np
import math
from dataclasses import dataclass
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.neighbors import NearestNeighbors
from sklearn.linear_model import Ridge

# =========================================================
# WEEK 13 — FUNCTION 6 (RL-AUGMENTED PCA + CLUSTER-AWARE BO)
#
# Key Week 13 upgrades (RL lens):
#  1) Multi-Armed Bandit (MAB) over (acquisition_strategy × candidate_source)
#     - epsilon-greedy with epsilon decaying as n grows (explore->exploit shift)
#  2) Q-learning style updates:
#     Q(a) <- (1-α)Q(a) + α[r + γ max_a' Q(a')]
#     Here reward r is surrogate-estimated (you can replace with real y feedback after eval)
#  3) "Self-play" rollouts:
#     run a few internal proposal/reward/update cycles to refine policy before final pick
#
# Goal: propose x_next in [0,1]^5 (<= 6 decimals) targeting local maxima.
# =========================================================

# -----------------------------
# 1) Data (as provided)
# -----------------------------
X_raw = np.array([
 [0.7281861, 0.15469257, 0.73255167, 0.69399651, 0.05640131],
 [0.24238435, 0.84409997, 0.5778091, 0.67902128, 0.50195289],
 [0.72952261, 0.7481062, 0.67977464, 0.35655228, 0.67105368],
 [0.77062024, 0.11440374, 0.04677993, 0.64832428, 0.27354905],
 [0.6188123, 0.33180214, 0.18728787, 0.75623847, 0.3288348],
 [0.78495809, 0.91068235, 0.7081201, 0.95922543, 0.0049115],
 [0.14511079, 0.8966846, 0.89632223, 0.72627154, 0.23627199],
 [0.94506907, 0.28845905, 0.97880576, 0.96165559, 0.59801594],
 [0.12572016, 0.86272469, 0.02854433, 0.24660527, 0.75120624],
 [0.75759436, 0.35583141, 0.0165229, 0.4342072, 0.11243304],
 [0.5367969, 0.30878091, 0.41187929, 0.38822518, 0.5225283],
 [0.95773967, 0.23566857, 0.09914585, 0.15680593, 0.07131737],
 [0.6293079, 0.80348368, 0.81140844, 0.04561319, 0.11062446],
 [0.02173531, 0.42808424, 0.83593944, 0.48948866, 0.51108173],
 [0.43934426, 0.69892383, 0.42682022, 0.10947609, 0.87788847],
 [0.25890557, 0.79367771, 0.6421139, 0.19667346, 0.59310318],
 [0.43216593, 0.71561781, 0.3418191, 0.70499988, 0.61496184],
 [0.78287982, 0.53633586, 0.44328356, 0.85969983, 0.01032599],
 [0.9217762, 0.93187122, 0.41487637, 0.59505727, 0.73562569],
 [0.12667892, 0.2914703, 0.06452848, 0.6805146, 0.89281919],
 [1.057739, 1.031871, 1.078805, 1.061655, 0.992819],
 [0.183405, 0.304243, 0.524756, 0.431945, 0.29123 ],
 [0.268807, 0.268756, 0.495982, 0.986904, 0.010463],
 [0.071886, 0.119564, 0.11427 , 0.97486 , 0.062381],
 [0.985053, 0.912856, 0.967723, 0.993021, 0.944062],
 [0.985053, 0.912856, 0.967723, 0.993021, 0.944062],
 [0.289326, 0.014308, 0.819185, 0.769045, 0.018673],
 [0.437318, 0.251829, 0.542272, 0.788906, 0.000000],
 [0.395320, 0.238886, 0.702458, 0.805605, 0.000000],
 [0.019634, 0.099529, 0.422099, 0.998212, 0.080632],
 [0.414000, 0.246000, 0.780000, 0.832000, 0.000000],
 [0.198714, 0.096701, 0.135299, 1.000000, 0.405497]
], dtype=float)

y_raw = np.array([
 -0.71426495, -1.20995524, -1.67219994, -1.53605771, -0.82923655,
 -1.24704893, -1.23378638, -1.69434344, -2.57116963, -1.30911635,
 -1.14478485, -1.91267714, -1.62283895, -1.35668211, -2.0184254,
 -1.70255784, -1.29424696, -0.93575656, -2.15576776, -1.74688209,
 -2.868905011263093, -0.9489046340640067, -0.6674108573004914,
 -1.3769650251311083, -2.5104529076756172, -2.5498833751068073,
 -0.7529123509459484, -0.36323892495106164, -0.29318472382773614,
 -1.1654038633763473, -0.36523877995026865, -1.52656338235905
], dtype=float)

assert X_raw.shape[0] == y_raw.shape[0]
assert X_raw.shape[1] == 5

# -----------------------------
# 2) Hygiene (clamp + dedup)
# -----------------------------
def clamp01(X):
    return np.clip(X, 0.0, 1.0)

def dedup_average(X, y, tol=0.0):
    if tol > 0:
        X_key = np.round(X / tol) * tol
    else:
        X_key = X.copy()
    keys = [row.tobytes() for row in X_key]
    buckets = {}
    for i, k in enumerate(keys):
        buckets.setdefault(k, []).append(i)

    X_new, y_new = [], []
    for _, idxs in buckets.items():
        X_new.append(X[idxs[0]])
        y_new.append(float(np.mean(y[idxs])))
    return np.array(X_new, dtype=float), np.array(y_new, dtype=float)

X_raw = clamp01(X_raw)
X_raw, y_raw = dedup_average(X_raw, y_raw, tol=0.0)

# -----------------------------
# 3) Scaling
# -----------------------------
x_scaler = StandardScaler()
X_scaled = x_scaler.fit_transform(X_raw)

# -----------------------------
# 4) PCA view + PC ascent direction (fast, stable)
# -----------------------------
def fit_pca_view(X_obs, y_obs, var_threshold=0.92, seed=123):
    Xs = x_scaler.transform(X_obs)
    pca = PCA(random_state=seed).fit(Xs)
    cum = np.cumsum(pca.explained_variance_ratio_)
    k = int(np.searchsorted(cum, var_threshold) + 1)
    k = int(np.clip(k, 2, X_obs.shape[1]))

    Z = pca.transform(Xs)[:, :k]
    ridge = Ridge(alpha=1.0, random_state=seed).fit(Z, y_obs)
    d = ridge.coef_.astype(float)
    if np.linalg.norm(d) < 1e-12:
        d = np.ones_like(d)
    d = d / (np.linalg.norm(d) + 1e-12)
    return pca, k, d

# -----------------------------
# 5) Cluster view (kept)
# -----------------------------
def cluster_best_centroid(X_obs, y_obs, k=4, seed=123):
    Xs = x_scaler.transform(X_obs)
    km = KMeans(n_clusters=k, random_state=seed, n_init=20).fit(Xs)
    labels = km.labels_
    # pick cluster by (best y, mean y)
    best_cluster = max(
        range(k),
        key=lambda c: (float(np.max(y_obs[labels == c])), float(np.mean(y_obs[labels == c])))
    )
    centroid_scaled = km.cluster_centers_[best_cluster]
    centroid = x_scaler.inverse_transform(centroid_scaled.reshape(1, -1)).ravel()
    return np.clip(centroid, 0.0, 1.0), int(best_cluster)

# -----------------------------
# 6) Fast surrogate: bootstrapped Ridge ensemble
#     (Week 13: make feedback loops cheap -> faster convergence)
# -----------------------------
@dataclass(frozen=True)
class RidgeEnsembleCfg:
    n_ensemble: int = 25
    alpha: float = 1.0
    seed: int = 123

def fit_ridge_ensemble(X_obs, y_obs, cfg: RidgeEnsembleCfg):
    Xs = x_scaler.transform(X_obs)
    n = Xs.shape[0]
    models = []
    for m in range(cfg.n_ensemble):
        rng = np.random.default_rng(cfg.seed + 10_000 + m)
        boot = rng.integers(0, n, size=n)
        reg = Ridge(alpha=cfg.alpha, random_state=cfg.seed + m).fit(Xs[boot], y_obs[boot])
        models.append(reg)
    return models

def predict_ridge_ensemble(models, X_cand):
    Xs = x_scaler.transform(X_cand)
    preds = np.stack([m.predict(Xs) for m in models], axis=0)  # [E, N]
    mu = preds.mean(axis=0)
    sigma = preds.std(axis=0)
    return mu, sigma, preds

# -----------------------------
# 7) Acquisition helpers
# -----------------------------
def qei_mc(preds_members, y_best, xi=0.0, n_mc=64, seed=123):
    rng = np.random.default_rng(seed)
    E, N = preds_members.shape
    idx = rng.integers(0, E, size=(n_mc, N))
    samples = preds_members[idx, np.arange(N)[None, :]]
    improv = np.maximum(samples - (y_best + xi), 0.0)
    return improv.mean(axis=0)

def diversity_penalty(dmin, scale=0.02):
    return np.exp(-dmin / max(scale, 1e-9))

# -----------------------------
# 8) Candidate generators (PCA-guided)
# -----------------------------
def pca_candidates(pca, k, n, z_clip=2.1, seed=123):
    rng = np.random.default_rng(seed)
    Z = rng.uniform(-z_clip, z_clip, size=(n, k))
    Xs = (Z @ pca.components_[:k, :]) + pca.mean_
    X = x_scaler.inverse_transform(Xs)
    return np.clip(X, 0.0, 1.0)

def pca_trust(pca, k, z_center, n, sigma=0.30, seed=123):
    rng = np.random.default_rng(seed)
    Z = z_center.reshape(1, -1) + rng.normal(0.0, sigma, size=(n, k))
    Xs = (Z @ pca.components_[:k, :]) + pca.mean_
    X = x_scaler.inverse_transform(Xs)
    return np.clip(X, 0.0, 1.0)

# -----------------------------
# 9) RL policy (MAB + Q-learning)
# -----------------------------
ACQ_ARMS = ["qEI", "UCB", "UNC"]  # acquisition arms
SRC_ARMS = ["PCA_GLOBAL", "PCA_BEST", "PCA_CLUSTER", "PC_STEP"]  # source arms

def epsilon_schedule(n, eps_max=0.22, eps_min=0.05, n_ref=12, power=0.7):
    # more data => smaller epsilon (exploit more)
    return float(np.clip(eps_max * (n_ref / max(n, 1.0)) ** power, eps_min, eps_max))

def softmax(x, temp=0.35):
    x = np.asarray(x, dtype=float)
    x = x - np.max(x)
    ex = np.exp(x / max(temp, 1e-9))
    return ex / (np.sum(ex) + 1e-12)

def q_update(Q, a, r, alpha=0.6, gamma=0.2):
    # one-step bootstrapped Q-learning update
    target = r + gamma * float(np.max(Q))
    Q[a] = (1 - alpha) * Q[a] + alpha * target
    return Q

# -----------------------------
# 10) Proposal (Week 13)
# -----------------------------
def propose_next_point_week13(X_obs, y_obs, seed=123):
    rng = np.random.default_rng(seed)

    n = X_obs.shape[0]
    y_best = float(np.max(y_obs))
    y_min = float(np.min(y_obs))
    y_range = max(y_best - y_min, 1e-9)

    # best observed point
    x_best = X_obs[int(np.argmax(y_obs))]

    # PCA + ascent direction
    pca, k_pcs, pc_dir = fit_pca_view(X_obs, y_obs, var_threshold=0.92, seed=seed)

    # cluster best centroid
    x_cluster, _ = cluster_best_centroid(X_obs, y_obs, k=4, seed=seed)

    # PC coordinates
    z_best = pca.transform(x_scaler.transform(x_best.reshape(1, -1)))[:, :k_pcs].ravel()
    z_cluster = pca.transform(x_scaler.transform(x_cluster.reshape(1, -1)))[:, :k_pcs].ravel()

    # adaptive spacing (density-aware)
    nn2 = NearestNeighbors(n_neighbors=2).fit(X_obs)
    d2, _ = nn2.kneighbors(X_obs)
    p10 = float(np.quantile(d2[:, 1], 0.10))
    min_dist = float(np.clip(0.35 * p10, 8e-4, 4e-3))

    # candidate pools (smaller but efficient + RL decides where to focus)
    X_global = pca_candidates(pca, k_pcs, n=1200, z_clip=2.1, seed=seed + 1)
    X_best  = pca_trust(pca, k_pcs, z_best,    n=900,  sigma=0.26, seed=seed + 2)
    X_clust = pca_trust(pca, k_pcs, z_cluster, n=700,  sigma=0.32, seed=seed + 3)

    # structured PC-step exploitation (AlphaGo-like “policy improvement”)
    step_grid = np.linspace(0.05, 0.55, 11)
    Z_steps = np.array([z_best + t * pc_dir for t in step_grid], dtype=float)
    Xs_steps = (Z_steps @ pca.components_[:k_pcs, :]) + pca.mean_
    X_steps = np.clip(x_scaler.inverse_transform(Xs_steps), 0.0, 1.0)

    pools = {
        "PCA_GLOBAL": X_global,
        "PCA_BEST": X_best,
        "PCA_CLUSTER": X_clust,
        "PC_STEP": X_steps
    }

    # filter near-duplicates per pool
    nn1 = NearestNeighbors(n_neighbors=1).fit(X_obs)
    for kname in list(pools.keys()):
        dmin = nn1.kneighbors(pools[kname])[0].ravel()
        pools[kname] = pools[kname][dmin >= min_dist]

    # fit surrogate (fast feedback loop)
    ens_cfg = RidgeEnsembleCfg(n_ensemble=25, alpha=1.0, seed=seed)
    ens_models = fit_ridge_ensemble(X_obs, y_obs, ens_cfg)

    # RL: Q tables over (acq, src)
    Q = np.zeros((len(ACQ_ARMS), len(SRC_ARMS)), dtype=float)

    # schedules
    eps = epsilon_schedule(n)
    temp = float(np.clip(0.65 * (12 / max(n, 1.0)) ** 0.7, 0.18, 0.65))  # exploration in softmax
    xi = float(0.012 * y_range * (12 / max(n, 1.0)) ** 0.7)              # optimistic early, smaller later
    beta = float(2.3 + 1.7 * (12 / max(n, 1.0)) ** 0.7)

    # "self-play" rollouts: propose->reward->update a few times
    rollouts = 6
    best_pick = None
    best_score = -1e18

    for t in range(rollouts):
        # pick (acq, src) using epsilon-greedy + softmax over Q
        if rng.random() < eps:
            a_i = int(rng.integers(0, len(ACQ_ARMS)))
            s_i = int(rng.integers(0, len(SRC_ARMS)))
        else:
            # softmax sampling over Q as stochastic policy (helps avoid premature local traps)
            flat = Q.reshape(-1)
            pi = softmax(flat, temp=temp)
            choice = int(rng.choice(len(flat), p=pi))
            a_i = choice // len(SRC_ARMS)
            s_i = choice % len(SRC_ARMS)

        acq = ACQ_ARMS[a_i]
        src = SRC_ARMS[s_i]
        X_cand = pools[src]
        if X_cand.shape[0] == 0:
            # fallback: take from PCA_BEST if pool empty
            src = "PCA_BEST"
            s_i = SRC_ARMS.index(src)
            X_cand = pools[src]

        mu, sig, members = predict_ridge_ensemble(ens_models, X_cand)
        dmin = nn1.kneighbors(X_cand)[0].ravel()
        pen = diversity_penalty(dmin, scale=0.02)

        if acq == "qEI":
            qei = qei_mc(members, y_best=y_best, xi=xi, n_mc=64, seed=seed + 99 + t)
            score = qei * (1.0 - 0.25 * pen)
            idx = int(np.argmax(score))
            # reward proxy: normalized improvement estimate
            r = float(qei[idx] / (y_range + 1e-9) - 0.10 * pen[idx])

        elif acq == "UCB":
            score = (mu + beta * sig) - 0.05 * pen * y_range
            idx = int(np.argmax(score))
            r = float((score[idx] - y_best) / (y_range + 1e-9))

        else:  # "UNC"
            score = sig - 0.10 * pen
            idx = int(np.argmax(score))
            r = float(score[idx] / (np.median(sig) + 1e-9))

        # update Q (Q-learning)
        Q[a_i, s_i] = q_update(Q[a_i, :], s_i, r, alpha=0.6, gamma=0.2)[s_i]

        # track best overall pick during rollouts
        if float(score[idx]) > best_score:
            best_score = float(score[idx])
            best_pick = X_cand[idx].copy()

    # final selection: greedy on learned Q, then pick best candidate under that arm
    a_i, s_i = np.unravel_index(np.argmax(Q), Q.shape)
    acq = ACQ_ARMS[a_i]
    src = SRC_ARMS[s_i]
    X_cand = pools[src] if pools[src].shape[0] > 0 else pools["PCA_BEST"]

    mu, sig, members = predict_ridge_ensemble(ens_models, X_cand)
    dmin = nn1.kneighbors(X_cand)[0].ravel()
    pen = diversity_penalty(dmin, scale=0.02)

    if acq == "qEI":
        qei = qei_mc(members, y_best=y_best, xi=xi, n_mc=96, seed=seed + 777)
        score = qei * (1.0 - 0.25 * pen)
        idx = int(np.argmax(score))
    elif acq == "UCB":
        score = (mu + beta * sig) - 0.05 * pen * y_range
        idx = int(np.argmax(score))
    else:
        score = sig - 0.10 * pen
        idx = int(np.argmax(score))

    x_next = X_cand[idx]

    # safety fallback: if rollout best_pick exists and is better, use it
    if best_pick is not None:
        x_next = best_pick

    return np.clip(x_next, 0.0, 1.0)

# -----------------------------
# 11) Main (prints x_next only)
# -----------------------------
def main():
    x_next = propose_next_point_week13(X_raw, y_raw, seed=123)
    x_next_6 = np.round(x_next, 6)

    print("\n================ WEEK 13 FUNCTION 6 — NEXT DATA POINT (SUBMIT THIS) ================\n")
    print("x_next =", x_next_6)

if __name__ == "__main__":
    main()


================ WEEK 13 FUNCTION 6 — NEXT DATA POINT (SUBMIT THIS) ================

x_next = [0.526359 0.276911 0.849154 0.945463 0.      ]


C:\Anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(
